# Primary Beam Annulus Statistics

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/processing_functions_tutorials/image_analysis/primary_beam_annulus_statistics.ipynb)

The CASA pipeline's image quality assessment measures the noise of an image as the RMS of the restored image, not divided by the primary beam, where the primary beam lies between two levels (0.2 and 0.3 by default) and outside the clean mask, after Chauvenet clipping, channel by channel; then it takes the median over the continuum channels. Three functions measure it:

- `primary_beam_annulus_limits` gives the two primary beam levels of an image (pipeline `pblimits`).
- `calculate_primary_beam_annulus_statistics` gives the clipped statistics of every image plane in the annulus, once in the whole annulus and once outside the clean mask (pipeline `ia.statistics(algorithm="chauvenet", maxiter=5)`).
- `summarize_primary_beam_annulus_rms` gives the median, minimum and maximum of the RMS over the channels the caller selects (pipeline `image_rms`, `image_rms_min`, `image_rms_max`).

The levels and the channels are the caller's choice, and nothing here scores the result. `IMAGE_QUALITY_METRICS.md` at the repository root maps the functions to the pipeline code and lists where they depart from it.

## Install AstroVIPER

Installs AstroVIPER only where it is missing.

In [ ]:
import os
from importlib.metadata import PackageNotFoundError, version

try:
    print("Using astroviper version", version("astroviper"))
except PackageNotFoundError:
    os.system("pip install --upgrade astroviper[all]")

## API

In [ ]:
from astroviper.processing_functions.image_analysis import (
    calculate_primary_beam_annulus_statistics,
    primary_beam_annulus_limits,
    summarize_primary_beam_annulus_rms,
)

calculate_primary_beam_annulus_statistics?

## What the functions measure

1. **Levels.** The nominal levels, unless the primary beam at the edge of the image is above the nominal lower level: then the primary beam values `n_m // 20` and `2 * (n_m // 20)` pixels from the edge, read along `m` in the central column of the middle channel.
2. **Annulus.** In every `(time, frequency, polarization)` plane, the finite pixels where that plane's primary beam lies strictly between the two levels.
3. **Clean mask.** Each plane is measured twice: in the whole annulus, and outside the clean mask. With `mask_union=True` every plane excludes the union of the clean masks of all planes, which is what the pipeline excludes from a cube.
4. **Clipping.** Chauvenet's criterion with five iterations removes the outliers, and the pixels it keeps give the number of pixels, the RMS, the standard deviation, the median and the median absolute deviation, as CASA reports them.
5. **Summary.** The RMS outside the mask, or that of the whole annulus when the median over all channels of the pixels kept outside the mask is below 10 (the pipeline's rule for a voluminous mask), reduced to its median, minimum and maximum over the selected channels. A channel without kept pixels counts as an RMS of 0.

## Simulate a cube

A Stokes I cube of 256 x 256 pixels and 16 channels: a Gaussian primary beam that narrows with frequency, noise that rises across the band, a continuum source at the phase centre, a line source in the annulus in channels 6 to 9, and a compact source in the annulus that the clean mask misses. The clean mask covers the continuum source in every channel and the line source in its channels. The image is the restored image, so the sources are attenuated by the primary beam and the noise is flat across the field.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

N_PIXELS = 256
N_CHANNELS = 16
CELL = np.deg2rad(0.2 / 3600.0)
LINE_CHANNELS = np.arange(6, 10)

offsets = np.arange(N_PIXELS) - N_PIXELS // 2
dl, dm = np.meshgrid(offsets, offsets, indexing="ij")
frequency = 230.0e9 + 2.0e6 * np.arange(N_CHANNELS)
fwhm = 150.0 * frequency[0] / frequency
primary_beam = np.exp(-4.0 * np.log(2.0) * (dl**2 + dm**2) / fwhm[:, None, None] ** 2)


def gaussian(offset_l, offset_m, sigma):
    return np.exp(-((dl - offset_l) ** 2 + (dm - offset_m) ** 2) / (2.0 * sigma**2))


rng = np.random.default_rng(7)
noise = 1.0e-3 * np.linspace(1.0, 1.5, N_CHANNELS)
sky = rng.normal(0.0, 1.0, primary_beam.shape) * noise[:, None, None]
sky += 0.05 * primary_beam * gaussian(0, 0, 3.0)
sky[LINE_CHANNELS] += 0.02 * gaussian(75, 74, 3.0)
sky += 0.015 * gaussian(-95, -40, 1.5)

clean_mask = np.zeros(primary_beam.shape, dtype=bool)
clean_mask[:] = np.hypot(dl, dm) < 20
clean_mask[LINE_CHANNELS] |= np.hypot(dl - 75, dm - 74) < 10

The functions take an image dataset in memory on `(time, frequency, polarization, l, m)`. Its data group names the restored image (`sky`), its primary beam (`primary_beam`) and, optionally, the clean mask (`mask`, bool, `True` inside). The image must not be divided by the primary beam, and the mask must be a clean mask: a primary beam mask in that role would be excluded like one.

In [ ]:
def image_dataset(sky, primary_beam, mask):
    dims = ("time", "frequency", "polarization", "l", "m")
    variables = {"SKY_RESTORED": sky, "PRIMARY_BEAM": primary_beam, "MASK": mask}
    n_l, n_m = sky.shape[-2:]
    return xr.Dataset(
        {
            name: (dims, values[np.newaxis, :, np.newaxis])
            for name, values in variables.items()
        },
        coords={
            "time": [0.0],
            "frequency": frequency,
            "polarization": ["I"],
            "l": CELL * (n_l // 2 - np.arange(n_l)),
            "m": CELL * (np.arange(n_m) - n_m // 2),
        },
        attrs={
            "data_groups": {
                "restored": {
                    "sky": "SKY_RESTORED",
                    "primary_beam": "PRIMARY_BEAM",
                    "mask": "MASK",
                }
            }
        },
    )


image = image_dataset(
    sky.astype(np.float32), primary_beam.astype(np.float32), clean_mask
)

## Levels

The primary beam at the edge of this image is below 0.2, so the levels are the nominal ones.

In [ ]:
limits = primary_beam_annulus_limits(image, (0.2, 0.3))
limits

In an image of 64 x 64 pixels the edge lies in the main lobe of the primary beam, and the levels come from the primary beam, 3 and 6 pixels from the edge. The pipeline writes the levels into its mask expressions with six decimals; passing them rounded reproduces it.

In [ ]:
inner = slice(96, 160)
small_image = image_dataset(
    sky[:, inner, inner].astype(np.float32),
    primary_beam[:, inner, inner].astype(np.float32),
    clean_mask[:, inner, inner],
)
small_limits = primary_beam_annulus_limits(small_image, (0.2, 0.3))
small_limits, tuple(round(level, 6) for level in small_limits)

## Statistics of every plane

In [ ]:
statistics = calculate_primary_beam_annulus_statistics(image, limits, mask_union=True)
statistics

In [ ]:
table = statistics[["n_pixels", "n_pixels_outside_mask", "rms", "rms_outside_mask"]]
table = table.squeeze(drop=True).to_dataframe()
table[["rms", "rms_outside_mask"]] *= 1.0e3
table["noise"] = noise * 1.0e3
table.round(4).rename(
    columns={
        "rms": "rms (mJy)",
        "rms_outside_mask": "rms outside mask (mJy)",
        "noise": "noise (mJy)",
    }
)

In the whole annulus, Chauvenet's criterion removes most of the line source in channels 6 to 9, which the union of the clean masks excludes from the other variant. Outside the masks it removes the compact source the clean mask misses. The RMS of channel 7 outside the union of the clean masks, without and with the clipping:

In [ ]:
plane = image.isel(time=0, frequency=7, polarization=0)
beam = plane.PRIMARY_BEAM.values.astype(np.float64)
outside = (beam > limits[0]) & (beam < limits[1]) & ~clean_mask.any(axis=0)
values = plane.SKY_RESTORED.values[outside].astype(np.float64)
print(f"without clipping: {1e3 * np.sqrt(np.mean(values**2)):.3f} mJy")
print(f"after clipping: {1e3 * statistics.rms_outside_mask.values[0, 7, 0]:.3f} mJy")

In [ ]:
figure, (image_axis, rms_axis) = plt.subplots(
    1, 2, figsize=(12, 5), constrained_layout=True
)
image_axis.imshow(1e3 * sky[7].T, origin="lower", cmap="gray_r", vmin=-4.0, vmax=8.0)
image_axis.contour(
    primary_beam[7].T, levels=list(limits), colors="tab:blue", linewidths=1.0
)
image_axis.contour(
    clean_mask.any(axis=0).T, levels=[0.5], colors="tab:red", linewidths=1.0
)
image_axis.set_title("channel 7: annulus (blue) and union of the clean masks (red)")
image_axis.set_xticks([])
image_axis.set_yticks([])

channels = np.arange(N_CHANNELS)
rms_axis.plot(channels, 1e3 * noise, "k--", label="simulated noise")
rms_axis.plot(
    channels, 1e3 * statistics.rms.values[0, :, 0], "o", label="whole annulus"
)
rms_axis.plot(
    channels,
    1e3 * statistics.rms_outside_mask.values[0, :, 0],
    "x",
    label="outside the clean masks",
)
rms_axis.axvspan(
    LINE_CHANNELS[0] - 0.5, LINE_CHANNELS[-1] + 0.5, color="0.9", label="line channels"
)
rms_axis.set_xlabel("channel")
rms_axis.set_ylabel("clipped RMS (mJy)")
rms_axis.legend()

With `mask_union=False` every plane excludes only its own clean mask, which is what a task holding a block of channels can compute: the line source region then counts in the channels without line emission.

In [ ]:
own_masks = calculate_primary_beam_annulus_statistics(image, limits)
own_masks.n_pixels_outside_mask.values.ravel()

## Summary over the continuum channels

The caller maps the continuum frequency ranges to channel indices, here the line-free channels 0 to 5 and 10 to 15. A repeated index counts repeatedly, as in the pipeline.

In [ ]:
continuum_channels = np.r_[0:6, 10:16]
summary = summarize_primary_beam_annulus_rms(statistics, continuum_channels)
summary

When the clean mask leaves fewer than 10 annulus pixels per channel (the median over all channels), the summary takes the whole annulus instead, and `full_annulus` says so. Here the mask leaves 9 pixels of the annulus uncovered, and the clipping keeps 7 to 9 of them in each channel.

In [ ]:
voluminous_mask = np.ones(primary_beam.shape, dtype=bool)
voluminous_mask[:, np.hypot(dl + 105, dm) < 1.5] = False
voluminous = calculate_primary_beam_annulus_statistics(
    image_dataset(
        sky.astype(np.float32), primary_beam.astype(np.float32), voluminous_mask
    ),
    limits,
    mask_union=True,
)
print(voluminous.n_pixels_outside_mask.values.ravel())
summarize_primary_beam_annulus_rms(voluminous, continuum_channels)

## Notes

- The pipeline's ALMA image score divides this RMS by the expected sensitivity. Scores, thresholds and the choice of levels and continuum channels are the caller's.
- Where the functions depart from the pipeline (the Chauvenet z-score, Stokes planes measured one by one, unmasked non-finite pixels, continuum images on `taylor_term`) is listed in `IMAGE_QUALITY_METRICS.md`.
- `tests/component/test_primary_beam_annulus_statistics_casa.py` compares the three functions with CASA `ia.statistics` and the pipeline's `pblimits`.